In [5]:
import pandas as pd
import csv
import pickle
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import os
import random
from itertools import product


from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search

In [6]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)

In [7]:
def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30) -> pd.DataFrame:
    """Return AR(1) innovations (residuals) for each column of X."""
    X_innov = pd.DataFrame(index=X.index, columns=X.columns, dtype="float64")

    for col in X.columns:
        s = pd.to_numeric(X[col], errors="coerce")
        tmp = pd.DataFrame({"x": s, "x_lag1": s.shift(1)}).dropna()
        if len(tmp) < min_obs or tmp["x"].nunique() < 3 or tmp["x_lag1"].nunique() < 3:
            continue

        res = sm.OLS(tmp["x"], sm.add_constant(tmp["x_lag1"])).fit()
        X_innov.loc[tmp.index, col] = res.resid

    return X_innov

In [9]:
random.seed(42)
# create feature matrix
X = feature_matrix.copy()
random.seed(42)

# Separate topic (name) and stock (numeric) columns
topic_cols = [col for col in X.columns if not str(col).isdigit()]
stock_cols = [col for col in X.columns if str(col).isdigit()]

# # User-selected number of each (or use len(...) for "all")
# num_topics = len(topic_cols)  # e.g., 100
# num_stocks = 0

# take random sample of 200 stocks from y 
sampled_stock_cols = random.sample(stock_cols, 200)
y = y[sampled_stock_cols]

# # Final filtered dataframe
# X = X[selected_topics + selected_stocks]

# # Convert stock returns to log returns: log(1+r)
# X[selected_stocks] = np.log(X[selected_stocks] + 1)

# # ensure that all indices align
# common_index = X.index.intersection(y.index)
# X = X.loc[common_index]
# y = y.loc[common_index]

In [10]:
import pandas as pd
import numpy as np
from grid_search import estimate_single_config

def run_all_y_columns(
    X_stocks, X_topics, window_size=40, n_lags=16, lambda_val=0.0093375
):
    all_summaries = []
    all_details = []

    for i in range(200):
        print(f"Running column {i}...")

        y = X_stocks.iloc[:, i]
        result = estimate_single_config(
            X=X_topics,
            y=y,
            window_size=window_size,
            n_lags=n_lags,
            lambda_val=lambda_val
        )

        summary = pd.DataFrame([result['summary']])
        summary['target_column'] = X_stocks.columns[i]
        all_summaries.append(summary)

        details = result['details'].copy()
        details['target_column'] = X_stocks.columns[i]
        all_details.append(details)

    summaries_df = pd.concat(all_summaries, ignore_index=True)
    details_df = pd.concat(all_details, ignore_index=True)

    return summaries_df, details_df


In [ ]:
summaries, details = run_first_100_y_columns(X_stocks, X_topics)

print("All Summaries:")
print(summaries.head())

print("\nAll Details:")
print(details.head())

Running column 90105...
Running column 34948...
Running column 13856...
Running column 92044...
Running column 75316...
Running column 70228...
Running column 64929...
Running column 45495...
Running column 91973...


c:\Users\jonat\Lasso_paper\Empirical\scripts\lasso_11_2025\stage2.py:24: RuntimeWarning: invalid value encountered in log
  return np.log(1 - kappa * np.exp(pred_t)) - np.log(1 - kappa * np.exp(pred_t1)) + intercept


Running column 30681...
Running column 90689...
Running column 92010...
Running column 88593...
Running column 27430...
Running column 89438...
Running column 81665...
Running column 16548...
Running column 15202...
Running column 28118...
Running column 64186...
Running column 66683...
Running column 86128...
Running column 89630...
Running column 13936...
Running column 89049...
Running column 60186...
Running column 91525...
Running column 90305...
Running column 91119...
Running column 81521...
Running column 64390...
Running column 83835...
Running column 89428...
Running column 75346...
Running column 10933...
Running column 92406...
Running column 50876...
Running column 91065...
Running column 81675...
Running column 77818...
Running column 49680...
Running column 63263...
Running column 92543...
Running column 77730...
Running column 27983...
Running column 79668...
Running column 28564...
Running column 79037...
Running column 77928...
Running column 89649...
Running column 7

In [12]:
pd.DataFrame.to_parquet(summaries, 'stage1_stage2_summaries.parquet')
pd.DataFrame.to_parquet(details, 'stage1_stage2_details.parquet')

In [ ]:
summaries = pd.read_parquet('stage1_stage2_summaries.parquet')
details = pd.read_parquet('stage1_stage2_details.parquet')